# Fine-Tuning BERT for Question Answering

This notebook walks through fine-tuning `bert-base-uncased` on a **Question Answering** task using the Hugging Face ecosystem.

Supports two dataset modes:
- **Option A** — Load a dataset directly from the Hugging Face Hub (e.g. SQuAD)
- **Option B** — Load from a custom local CSV file

---
**Requirements:** `transformers`, `datasets`, `torch`, `accelerate`, `evaluate`

## 1. Install Dependencies

In [ ]:
!pip install transformers datasets evaluate accelerate torch --quiet

## 2. Imports & Configuration

In [ ]:
import os
import json
import torch
import numpy as np
import pandas as pd
from pathlib import Path

from datasets import Dataset, DatasetDict, load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForQuestionAnswering,
    TrainingArguments,
    Trainer,
    DefaultDataCollator,
)
import evaluate

print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device    : {device}")

In [ ]:
# ── Global Config ─────────────────────────────────────────────────────────────
MODEL_NAME        = "bert-base-uncased"   # Base checkpoint
OUTPUT_DIR        = "./bert-qa-finetuned" # Where the model is saved
MAX_LENGTH        = 384                   # Max token length per sample
DOC_STRIDE        = 128                   # Overlap when context is split
BATCH_SIZE        = 16
NUM_EPOCHS        = 3
LEARNING_RATE     = 2e-5
WEIGHT_DECAY      = 0.01
WARMUP_RATIO      = 0.1
SEED              = 42

# ── Dataset source: switch between 'hub' and 'csv' ────────────────────────────
DATASET_SOURCE    = "hub"                 # 'hub' | 'csv'

# Hub settings (used when DATASET_SOURCE == 'hub')
HUB_DATASET_NAME  = "squad"              # Any HF QA dataset
HUB_DATASET_CONFIG = None               # e.g. 'v2' for SQuAD 2.0

# CSV settings (used when DATASET_SOURCE == 'csv')
CSV_TRAIN_PATH    = "train.csv"
CSV_VAL_PATH      = "validation.csv"
# Required CSV columns: id, context, question, answers
# 'answers' must be a JSON string: {"text": ["..."], "answer_start": [42]}

print("Config loaded ✓")

## 3. Load Dataset

### Option A — Hugging Face Hub
Set `DATASET_SOURCE = "hub"` and pick any QA dataset (default: SQuAD).

### Option B — Custom CSV
Set `DATASET_SOURCE = "csv"` and point `CSV_TRAIN_PATH` / `CSV_VAL_PATH` to your files.

**Required CSV columns:**
| Column | Type | Example |
|---|---|---|
| `id` | str | "sample_001" |
| `context` | str | "The Eiffel Tower is in Paris..." |
| `question` | str | "Where is the Eiffel Tower?" |
| `answers` | JSON str | `{"text": ["Paris"], "answer_start": [27]}` |

In [ ]:
def load_csv_dataset(train_path: str, val_path: str) -> DatasetDict:
    """Load a QA dataset from CSV files and convert to HF DatasetDict."""
    def parse_csv(path):
        df = pd.read_csv(path)
        required = {"id", "context", "question", "answers"}
        missing = required - set(df.columns)
        if missing:
            raise ValueError(f"CSV missing columns: {missing}")
        # Parse 'answers' from JSON string → dict
        df["answers"] = df["answers"].apply(
            lambda x: json.loads(x) if isinstance(x, str) else x
        )
        return Dataset.from_pandas(df[["id", "context", "question", "answers"]])

    return DatasetDict({
        "train":      parse_csv(train_path),
        "validation": parse_csv(val_path),
    })


# ── Load based on selected source ─────────────────────────────────────────────
if DATASET_SOURCE == "hub":
    print(f"Loading '{HUB_DATASET_NAME}' from Hugging Face Hub...")
    raw_datasets = load_dataset(HUB_DATASET_NAME, HUB_DATASET_CONFIG)
elif DATASET_SOURCE == "csv":
    print(f"Loading CSV dataset from '{CSV_TRAIN_PATH}' / '{CSV_VAL_PATH}'...")
    raw_datasets = load_csv_dataset(CSV_TRAIN_PATH, CSV_VAL_PATH)
else:
    raise ValueError(f"Unknown DATASET_SOURCE: '{DATASET_SOURCE}'. Use 'hub' or 'csv'.")

print(raw_datasets)
print("\nSample record:")
print(raw_datasets["train"][0])

## 4. Tokenizer Setup

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f"Tokenizer loaded: {MODEL_NAME}")
print(f"Vocab size: {tokenizer.vocab_size:,}")

## 5. Preprocessing — Tokenize with Answer Span Mapping

For QA, we must map each answer's character position to a **start / end token index**. When the context is longer than `MAX_LENGTH`, it is split into overlapping windows using `DOC_STRIDE`.

In [ ]:
def preprocess_training_examples(examples):
    questions = [q.strip() for q in examples["question"]]
    inputs = tokenizer(
        questions,
        examples["context"],
        max_length=MAX_LENGTH,
        truncation="only_second",
        stride=DOC_STRIDE,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )

    offset_mapping   = inputs.pop("offset_mapping")
    sample_map       = inputs.pop("overflow_to_sample_mapping")
    answers          = examples["answers"]
    start_positions  = []
    end_positions    = []

    for i, offset in enumerate(offset_mapping):
        sample_idx   = sample_map[i]
        answer       = answers[sample_idx]
        input_ids    = inputs["input_ids"][i]
        cls_index    = input_ids.index(tokenizer.cls_token_id)

        sequence_ids = inputs.sequence_ids(i)

        # Locate context token span
        ctx_start = next(j for j, s in enumerate(sequence_ids) if s == 1)
        ctx_end   = len(sequence_ids) - 1
        while sequence_ids[ctx_end] != 1:
            ctx_end -= 1

        # If no answers or answer not in window → CLS token
        if len(answer["answer_start"]) == 0:
            start_positions.append(cls_index)
            end_positions.append(cls_index)
            continue

        char_start = answer["answer_start"][0]
        char_end   = char_start + len(answer["text"][0])

        if (offset[ctx_start][0] > char_end or
                offset[ctx_end][1] < char_start):
            start_positions.append(cls_index)
            end_positions.append(cls_index)
            continue

        # Walk to correct token positions
        tok_start = ctx_start
        while tok_start <= ctx_end and offset[tok_start][0] <= char_start:
            tok_start += 1
        start_positions.append(tok_start - 1)

        tok_end = ctx_end
        while tok_end >= ctx_start and offset[tok_end][1] >= char_end:
            tok_end -= 1
        end_positions.append(tok_end + 1)

    inputs["start_positions"] = start_positions
    inputs["end_positions"]   = end_positions
    return inputs


def preprocess_validation_examples(examples):
    """Validation keeps offset_mapping for answer extraction."""
    questions = [q.strip() for q in examples["question"]]
    inputs = tokenizer(
        questions,
        examples["context"],
        max_length=MAX_LENGTH,
        truncation="only_second",
        stride=DOC_STRIDE,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )
    sample_map = inputs.pop("overflow_to_sample_mapping")
    example_ids = []

    for i in range(len(inputs["input_ids"])):
        sample_idx = sample_map[i]
        example_ids.append(examples["id"][sample_idx])
        sequence_ids = inputs.sequence_ids(i)
        inputs["offset_mapping"][i] = [
            (o if sequence_ids[k] == 1 else None)
            for k, o in enumerate(inputs["offset_mapping"][i])
        ]

    inputs["example_id"] = example_ids
    return inputs


print("Preprocessing functions defined ✓")

In [ ]:
train_dataset = raw_datasets["train"].map(
    preprocess_training_examples,
    batched=True,
    remove_columns=raw_datasets["train"].column_names,
    desc="Tokenizing train set",
)

validation_dataset = raw_datasets["validation"].map(
    preprocess_validation_examples,
    batched=True,
    remove_columns=raw_datasets["validation"].column_names,
    desc="Tokenizing validation set",
)

# Trainer needs a clean version without extra columns
validation_dataset_for_trainer = validation_dataset.remove_columns(
    ["example_id", "offset_mapping"]
)

print(f"Train features    : {train_dataset.num_rows:,}")
print(f"Validation features: {validation_dataset.num_rows:,}")

## 6. Load Model

In [ ]:
model = AutoModelForQuestionAnswering.from_pretrained(MODEL_NAME)
model.to(device)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total params    : {total_params:,}")
print(f"Trainable params: {trainable_params:,}")

## 7. Metrics — Exact Match & F1

In [ ]:
metric = evaluate.load("squad")

def postprocess_qa_predictions(examples, features, raw_predictions, n_best=20, max_answer_length=30):
    """Convert raw logits to readable answer strings."""
    all_start_logits, all_end_logits = raw_predictions

    example_id_to_index = {ex["id"]: i for i, ex in enumerate(examples)}
    features_per_example = {}
    for i, feat in enumerate(features):
        eid = feat["example_id"]
        features_per_example.setdefault(eid, []).append(i)

    predictions = {}
    for example in examples:
        eid = example["id"]
        context = example["context"]
        min_null_score = None
        valid_answers  = []

        for feat_idx in features_per_example.get(eid, []):
            start_logits  = all_start_logits[feat_idx]
            end_logits    = all_end_logits[feat_idx]
            offset_mapping = features[feat_idx]["offset_mapping"]

            cls_index = features[feat_idx]["input_ids"].index(tokenizer.cls_token_id)
            null_score = start_logits[cls_index] + end_logits[cls_index]
            if min_null_score is None or null_score < min_null_score:
                min_null_score = null_score

            start_indices = np.argsort(start_logits)[-n_best:][::-1]
            end_indices   = np.argsort(end_logits)[-n_best:][::-1]

            for si in start_indices:
                for ei in end_indices:
                    if (offset_mapping[si] is None or
                            offset_mapping[ei] is None or
                            ei < si or
                            ei - si + 1 > max_answer_length):
                        continue
                    valid_answers.append({
                        "score": start_logits[si] + end_logits[ei],
                        "text" : context[offset_mapping[si][0]: offset_mapping[ei][1]],
                    })

        best = max(valid_answers, key=lambda x: x["score"]) if valid_answers else {"text": ""}
        predictions[eid] = best["text"]

    return predictions


def compute_metrics(eval_preds):
    preds = postprocess_qa_predictions(
        raw_datasets["validation"],
        validation_dataset,
        eval_preds.predictions,
    )
    formatted_preds = [{"id": k, "prediction_text": v} for k, v in preds.items()]
    references = [
        {"id": ex["id"], "answers": ex["answers"]}
        for ex in raw_datasets["validation"]
    ]
    return metric.compute(predictions=formatted_preds, references=references)


print("Metrics defined ✓")

## 8. Training Arguments

In [ ]:
training_args = TrainingArguments(
    output_dir                  = OUTPUT_DIR,
    num_train_epochs            = NUM_EPOCHS,
    per_device_train_batch_size = BATCH_SIZE,
    per_device_eval_batch_size  = BATCH_SIZE,
    learning_rate               = LEARNING_RATE,
    weight_decay                = WEIGHT_DECAY,
    warmup_ratio                = WARMUP_RATIO,
    eval_strategy               = "epoch",
    save_strategy               = "epoch",
    load_best_model_at_end      = True,
    metric_for_best_model       = "f1",
    greater_is_better           = True,
    logging_dir                 = f"{OUTPUT_DIR}/logs",
    logging_steps               = 100,
    fp16                        = torch.cuda.is_available(),  # Mixed precision on GPU
    seed                        = SEED,
    report_to                   = "none",
)

print(training_args)

## 9. Train

In [ ]:
trainer = Trainer(
    model           = model,
    args            = training_args,
    train_dataset   = train_dataset,
    eval_dataset    = validation_dataset_for_trainer,
    tokenizer       = tokenizer,
    data_collator   = DefaultDataCollator(),
    compute_metrics = compute_metrics,
)

print("Starting training...")
train_result = trainer.train()

# Save final model & tokenizer
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print(f"\nModel saved to '{OUTPUT_DIR}'")
print(train_result.metrics)

## 10. Evaluate on Validation Set

In [ ]:
eval_results = trainer.evaluate()
print("\n── Validation Results ──────────────────")
for k, v in eval_results.items():
    print(f"  {k:<30}: {v:.4f}" if isinstance(v, float) else f"  {k:<30}: {v}")

## 11. Inference — Test the Fine-Tuned Model

In [ ]:
from transformers import pipeline

qa_pipeline = pipeline(
    "question-answering",
    model     = OUTPUT_DIR,
    tokenizer = OUTPUT_DIR,
    device    = 0 if torch.cuda.is_available() else -1,
)

# ── Try your own examples here ────────────────────────────────────────────────
test_cases = [
    {
        "context" : "The Eiffel Tower is located in Paris, France. It was built in 1889 "
                    "as the entrance arch for the 1889 World's Fair.",
        "question": "Where is the Eiffel Tower located?",
    },
    {
        "context" : "Python was created by Guido van Rossum and first released in 1991. "
                    "It emphasises code readability and supports multiple programming paradigms.",
        "question": "Who created Python?",
    },
]

for tc in test_cases:
    result = qa_pipeline(question=tc["question"], context=tc["context"])
    print(f"Q : {tc['question']}")
    print(f"A : {result['answer']}  (score: {result['score']:.4f})")
    print()

## 12. Save Training Log & Push to Hub (Optional)

In [ ]:
# Save training metrics to JSON
log_path = Path(OUTPUT_DIR) / "training_results.json"
with open(log_path, "w") as f:
    json.dump({
        "train_metrics": train_result.metrics,
        "eval_metrics" : eval_results,
    }, f, indent=2)
print(f"Results saved to {log_path}")

In [ ]:
# ── Push to Hugging Face Hub (optional) ───────────────────────────────────────
# Uncomment and fill in your repo name to share the model publicly.

# from huggingface_hub import notebook_login
# notebook_login()                               # Paste your HF token when prompted

# trainer.push_to_hub("your-username/bert-qa-finetuned")

---
## Summary

| Step | Description |
|---|---|
| Model | `bert-base-uncased` |
| Task | Extractive Question Answering |
| Dataset | SQuAD (Hub) or custom CSV |
| Tokenization | Sliding window with stride for long contexts |
| Metrics | Exact Match (EM) & F1 |
| Output | Saved model + tokenizer in `./bert-qa-finetuned` |

### Key Hyperparameters to Tune
- `LEARNING_RATE` — try `1e-5` to `5e-5`
- `MAX_LENGTH` — increase to `512` for longer contexts
- `DOC_STRIDE` — smaller = more overlap, better coverage
- `NUM_EPOCHS` — 2–4 epochs is usually optimal for QA